# HW3 從零實作注意力與 GPT 模型

> 說明：<https://github.com/chang-ye-tu/genai/blob/main/hw/hw3.md>　取材：Raschka《Build a Large Language Model (From Scratch)》第 3、4 章（套件 `llms-from-scratch`）
> 核心段落：第 1–5 節（實作測驗只出這些段落的題目）；第 6 節為選做，不計分。
> 授權：本筆記本呼叫並改寫 Sebastian Raschka 的開源專案 <https://github.com/rasbt/LLMs-from-scratch>（Apache-2.0，© Sebastian Raschka）的範例程式；改寫部分（本課程的講解、問題與資料）同樣以 Apache-2.0 散布，完整授權文本見 repo 的 `LICENSES/Apache-2.0.txt`。
> 做法：**執行階段 → 變更執行階段類型 → T4 GPU**，由上而下逐格執行；看到「✍️ 請回答」就把觀察寫進該文字格。全部跑完後「檔案 → 下載 → .ipynb」上傳 iLearn，再作答實作測驗。
> **請勿更改模型名稱、版本、隨機種子與資料檔**，否則實作測驗的數值題會對不上。
> 本作業全部可在 CPU 執行；選 T4 只是為了第 4 節較快。


In [ ]:
# @title 第 0 節：安裝
%pip -q install --no-deps llms-from-scratch==1.0.19
%pip -q install tiktoken==0.14.0
import torch, tiktoken
torch.manual_seed(123)
import platform, importlib.metadata as _meta
def _v(p):
    try: return _meta.version(p)
    except Exception: return "missing"  # metadata 查不到時印 missing；若同一格更早的 import 已失敗，程式到不了這裡，check_submissions 會判「無版本資訊／執行錯誤」
print("VERSIONS", "python=" + platform.python_version(), "torch=" + torch.__version__, *[p + "=" + _v(p) for p in ["llms-from-scratch", "tiktoken"]])


## 第 1 節 最簡單的注意力：用內積找「相關的 token」

六個 token（"Your journey starts with one step"）各用一個 3 維向量表示。以第 2 個 token（journey）當 query：
1. 與每個 token 的向量做內積 → 注意力分數；2. softmax → 注意力權重（加總為 1）；3. 權重 × 各向量加總 → 上下文向量。
這正是第 4 單元投影片「計算相關性 → 加權和」的兩步。


In [ ]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89],  # Your
   [0.55, 0.87, 0.66],  # journey
   [0.57, 0.85, 0.64],  # starts
   [0.22, 0.58, 0.33],  # with
   [0.77, 0.25, 0.10],  # one
   [0.05, 0.80, 0.55]]) # step
query = inputs[1]
attn_scores_2 = inputs @ query
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
context_vec_2 = attn_weights_2 @ inputs
print("注意力分數：", [round(float(x), 4) for x in attn_scores_2])
print("注意力權重：", [round(float(x), 4) for x in attn_weights_2], "| 加總 =", round(float(attn_weights_2.sum()), 4))
print("上下文向量：", [round(float(x), 4) for x in context_vec_2])


In [ ]:
# 一次算完所有 token：6×6 的注意力權重矩陣
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1)
torch.set_printoptions(precision=3, sci_mode=False)
print(attn_weights)
print("每一列加總：", attn_weights.sum(dim=-1))


✍️ **請回答 1-1**：哪個 token 對「journey」的權重最高（除了它自己）？為什麼「starts」和「journey」權重接近？權重加總為 1 有什麼意義？

（在這裡作答）


## 第 2 節 可訓練的 Q、K、V 與因果遮罩

真正的注意力多了三個可訓練矩陣：query、key、value。語言模型只能看左邊（第 4 單元的 causal attention），做法是把右上三角的分數設成 −∞，softmax 後就是 0。


In [ ]:
from llms_from_scratch.ch03 import SelfAttention_v2
torch.manual_seed(789)
sa = SelfAttention_v2(d_in=3, d_out=2)
print("有 Q/K/V 之後的上下文向量：\n", sa(inputs))
queries, keys = sa.W_query(inputs), sa.W_key(inputs)
scores = queries @ keys.T
mask = torch.triu(torch.ones(6, 6), diagonal=1)
masked = scores.masked_fill(mask.bool(), -torch.inf)
weights = torch.softmax(masked / keys.shape[-1] ** 0.5, dim=-1)
print("因果遮罩後的注意力權重：\n", weights)
print("每列加總：", weights.sum(dim=-1))


✍️ **請回答 2-1**：遮罩後為什麼右上三角全是 0？為什麼要除以 sqrt(d_k) 再 softmax？這和第 1 單元「文字接龍只能看前面」有什麼關係？

（在這裡作答）


## 第 3 節 多頭注意力：GPT-2 尺寸

GPT-2 small 的設定：向量 768 維、12 個 head。算一算一個多頭注意力模組有幾個參數，並確認輸入輸出形狀。


In [ ]:
from llms_from_scratch.ch03 import MultiHeadAttention
torch.manual_seed(123)
mha = MultiHeadAttention(d_in=768, d_out=768, context_length=1024, dropout=0.0, num_heads=12, qkv_bias=True)
n_params = sum(p.numel() for p in mha.parameters())
print(f"一個多頭注意力模組的參數量：{n_params:,}")
x = torch.randn(2, 6, 768)  # batch=2, 6 個 token, 768 維
print("輸入形狀：", tuple(x.shape), "→ 輸出形狀：", tuple(mha(x).shape))


✍️ **請回答 3-1**：參數量怎麼來的？（提示：Q、K、V、輸出投影各是 768×768 的矩陣加偏差）為什麼輸出維度和輸入一樣？

（在這裡作答）


## 第 4 節 組裝 GPT-2 尺寸的模型（124M）：數參數

把「嵌入 → 12 個 Transformer block（注意力 + 前饋）→ 正規化 → 輸出層」組起來，就是 GPT-2 small 的結構。這裡沿用書中的 `GPT_CONFIG_124M`（`qkv_bias=False`）；原始 GPT-2 的 Q/K/V 有偏差項，HW5 載入官方權重時會改用 `qkv_bias=True`（共享權重後為 124,439,808 個參數）。


In [ ]:
from llms_from_scratch.ch04 import GPTModel
GPT_CONFIG_124M = {"vocab_size": 50257, "context_length": 1024, "emb_dim": 768,
                   "n_heads": 12, "n_layers": 12, "drop_rate": 0.1, "qkv_bias": False}
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
total = sum(p.numel() for p in model.parameters())
out_head = sum(p.numel() for p in model.out_head.parameters())
ffn = sum(p.numel() for b in model.trf_blocks for p in b.ff.parameters())
att = sum(p.numel() for b in model.trf_blocks for p in b.att.parameters())
print(f"總參數量：{total:,}")
print(f"若輸出層與詞嵌入共享權重（GPT-2 原始做法）：{total - out_head:,}")
print(f"12 個前饋層合計：{ffn:,} | 12 個注意力模組合計：{att:,}")
print(f"詞嵌入：{model.tok_emb.weight.numel():,} | 位置嵌入：{model.pos_emb.weight.numel():,}")
print(f"fp32 佔用記憶體：{total * 4 / 1024 / 1024:.2f} MiB（十進位約 {total * 4 / 1e6:.0f} MB）")


In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
batch = torch.stack([torch.tensor(tokenizer.encode("Every effort moves you")),
                     torch.tensor(tokenizer.encode("Every day holds a"))])
with torch.no_grad():
    logits = model(batch)
print("輸入形狀：", tuple(batch.shape), "→ logits 形狀：", tuple(logits.shape), "（每個位置對 50,257 個 token 各一個分數）")


✍️ **請回答 4-1**：為什麼前饋層的參數比注意力多一倍？「權重共享」省下的參數等於哪一個矩陣的大小？logits 最後一維為什麼是 50,257？

（在這裡作答）


## 第 5 節 沒訓練過的 GPT 會說什麼

參數還是隨機的，用 greedy 接龍十個 token（`generate_text_simple` 每一步直接取 logits 的 argmax；不做 softmax 結果也一樣，因為 softmax 不改變大小順序）。


In [ ]:
from llms_from_scratch.ch04 import generate_text_simple
model.eval()
start = "Hello, I am"
idx = torch.tensor(tokenizer.encode(start)).unsqueeze(0)
out = generate_text_simple(model=model, idx=idx, max_new_tokens=10, context_size=GPT_CONFIG_124M["context_length"])
print("輸出 token 數：", out.shape[1], "\n輸出文字：", repr(tokenizer.decode(out.squeeze(0).tolist())))


✍️ **請回答 5-1**：輸出為什麼是亂碼？架構已經和真正的 GPT-2 一樣了，還缺什麼？（HW5 會補上這一塊）

（在這裡作答）


## 第 6 節（選做）對照：HW1 的 Qwen2.5-1.5B

| | GPT-2 small (2019) | Qwen2.5-1.5B-Instruct (2024) |
|--|--|--|
| 層數 | 12 | 28 |
| 向量維度 | 768 | 1,536 |
| 注意力頭 | 12 | 12（KV 頭 2，GQA） |
| 詞彙表 | 50,257 | 151,936 |
| 上下文 | 1,024 | 32,768 |
| 參數 | 1.24 億 | 15.4 億 |


✍️ **請回答 6-1**：從表格看，五年間哪些數字變了最多？這些改變各對應到第 4、5 單元講的哪些概念（上下文長度、KV cache、多語詞彙表）？

（在這裡作答）


## ✍️ AI 使用聲明（必填）

| 項目 | 內容 |
|------|------|
| 使用的工具 | （例如：ChatGPT 免費版、Colab 內建 Gemini） |
| 用在哪些工作 | （例如：解釋錯誤訊息、幫我看懂某一格程式） |
| 我自己完成的部分 | （例如：全部執行、所有 ✍️ 回答） |
| 我如何驗證 AI 的說法 | （例如：實際執行、對照投影片） |

姓名／學號：
